# grayscale correlation
projecting down the results fromm 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset
3. predict results using NOFE and other methods
4. project to grayscale
5. evaluate correlation

---
### Utilities

In [1]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

using Device: mps
Using device: mps


----
### load model + dataset 

In [2]:
model, config = load_model(id='test1_model8_new_2026-04-16_t09-20-39' , model_type='regular')

In [3]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [106]:
date = "2019-01-10"

bbox = lon_min, lat_min, lon_max, lat_max

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

N=5000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method='ones')
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
X = sample.normed_values.detach().cpu().numpy()

total points available: 32761
generate data


---
## Get Predictions

In [107]:
# NOFE
pred_nofe = model(sample).detach().cpu().numpy()

In [108]:
# PCA
pca = PCA(n_components=3)
pred_pca = pca.fit_transform(X)

In [109]:
# t-SNE
pred_tsne = TSNE(
    n_components=3,
    perplexity=30,
    learning_rate="auto",
    random_state=42
).fit_transform(X)


In [110]:
# UMAP
umap_id = 'umap_buffer'
dir = f'{project_root}/ERA5/umap'
np.save(f'{dir}/{umap_id}.npy', X)
subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
pred_umap = np.load(f'{dir}/res_{umap_id}.npy')

os.remove(f'{dir}/{umap_id}.npy')
os.remove(f'{dir}/res_{umap_id}.npy')

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



---
## Grayscale Projection

In [111]:
pca = PCA(n_components=1)
grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 

In [112]:
pca = PCA(n_components=1)
grayscale_pca = pca.fit_transform(pred_pca).squeeze() 

In [113]:
pca = PCA(n_components=1)
grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 

In [114]:
# UMAP
pca = PCA(n_components=1)
grayscale_umap = pca.fit_transform(pred_umap).squeeze() 

---
## Correlate

In [115]:
correlations_nofe = np.array([
    np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [116]:
correlations_pca = np.array([
    np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [117]:
correlations_tsne = np.array([
    np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [118]:
correlations_umap = np.array([
    np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [119]:
corrs = pd.DataFrame({
    'nofe' : correlations_nofe,
    'pca' : correlations_pca,
    'tsne' : correlations_tsne,
    'umap' : correlations_umap
}, index=config['vars'])
print(corrs)
print()
print('Mean Values:')
print(f'NOFE:\t{corrs.nofe.mean():.2f}')
print(f'PCA:\t{corrs.pca.mean():.2f}')
print(f't-SNE:\t{corrs.tsne.mean():.2f}')
print(f'UMAP:\t{corrs.umap.mean():.2f}')

        nofe       pca      tsne      umap
w   0.274849  0.148615  0.028238  0.159814
v   0.120480  0.243853  0.009218  0.009234
z   0.359219  0.596990  0.738764  0.524445
q   0.690463  0.724003  0.289206  0.509799
r   0.616703  0.471509  0.301230  0.090930
t   0.337319  0.588065  0.890336  0.926821
d   0.130954  0.116282  0.001888  0.032231
o3  0.824222  0.770706  0.068695  0.080587
pv  0.775206  0.738262  0.165279  0.096535
u   0.033340  0.015102  0.256765  0.350490
vo  0.833701  0.845175  0.233099  0.301090

Mean Values:
NOFE:	0.45
PCA:	0.48
t-SNE:	0.27
UMAP:	0.28


---
# Run on full scale datset

In [ ]:
dataset = GraphLMDBReader(f'{project_root}/ERA5/datasets/regular/toyset_reg_fixcrop_ones_woPos_new_test.lmdb')
print(f'loaded {len(dataset)} samples')

loaded 100 samples


In [ ]:
corr_res_nofe = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # NOFE
    # =================== 
    pred_nofe = model(sample).detach().cpu().numpy()
    pca = PCA(n_components=1)
    grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 
    correlations_nofe = np.array([
        np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_nofe.append(correlations_nofe)

corr_res_nofe = np.vstack(corr_res_nofe)

In [ ]:
corr_res_pca = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # PCA
    # =================== 
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_pca = pca.fit_transform(pred_pca).squeeze() 
    correlations_pca = np.array([
        np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_pca.append(correlations_pca)

corr_res_pca = np.vstack(corr_res_pca)    

In [ ]:
corr_res_tsne = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # t-SNE
    # =================== 
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 
    correlations_tsne = np.array([
        np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_tsne.append(correlations_tsne)

corr_res_tsne = np.vstack(corr_res_tsne)

In [ ]:
corr_res_umap = []

#for s in range(len(reader)):
for s in range(10):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # UMAP
    # =================== 
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')
    pca = PCA(n_components=1)
    grayscale_umap = pca.fit_transform(pred_umap).squeeze() 
    correlations_umap = np.array([
        np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_umap.append(correlations_umap)

corr_res_umap = np.vstack(corr_res_umap)

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



... (8 outputs left out)

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



In [15]:
corrs = pd.DataFrame({
    'nofe' : corr_res_nofe.mean(axis=0),
    'pca' : corr_res_pca.mean(axis=0),
    'tsne' : corr_res_tsne.mean(axis=0),
    'umap' : corr_res_umap.mean(axis=0)
}, index=config['vars'])

print('Mean correlation per variable:')
print(corrs)

print('\nMean correlation coefficient:\n')
print(f'NOFE:\t{corr_res_nofe.mean():.2f}')
print(f'PCA:\t{corr_res_pca.mean():.2f}')
print(f't-SNE:\t{corr_res_tsne.mean():.2f}')
print(f'UMAP:\t{corr_res_umap.mean():.2f}')

Mean correlation per variable:
        nofe       pca      tsne      umap
w   0.414146  0.118097  0.059090  0.027627
v   0.198519  0.156512  0.087194  0.125037
z   0.164438  0.865105  0.865678  0.904047
q   0.717172  0.650944  0.436150  0.408532
r   0.594759  0.085261  0.176029  0.258329
t   0.279421  0.915569  0.905357  0.943589
d   0.042370  0.023404  0.027500  0.016275
o3  0.434209  0.137239  0.355621  0.356914
pv  0.644547  0.394935  0.187735  0.167060
u   0.096178  0.551097  0.436924  0.437557
vo  0.669722  0.444101  0.106886  0.141083

Mean correlation coefficient:

NOFE:	0.39
PCA:	0.39
t-SNE:	0.33
UMAP:	0.34
